In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Optional

# Input Models

@dataclass(frozen=True)
class IngestedTelemetryRecord:
    timestamp: any
    rack_temps: List[float]
    airflow_proxy: float
    chiller_power: float
    facility_power: float
    ups_load: float
    tariff: float
    raw_format: str

@dataclass
class ValidatedTelemetryRecord:
    data: IngestedTelemetryRecord
    is_valid: bool
    validation_flags: List[str]
    reason_codes: List[str]

@dataclass
class SystemStateRecord:
    """Historical context."""
    current_data: ValidatedTelemetryRecord
    last_safe_data: Optional[IngestedTelemetryRecord]
    delta_t_trend: float  # °C/min
    avg_rack_temp_15m: float
    avg_facility_power_1h: float
    is_stable: bool
    window_15m_count: int

# Output Model

class SafetyStatus(Enum):
    SAFE = "SAFE"
    BLOCKED = "BLOCKED"

@dataclass
class SafetyStatusRecord:
    """Safety decision record."""
    status: SafetyStatus
    reason_codes: List[str] = field(default_factory=list)
    description: str = ""

# Module Implementation

class EnerviaSafetyEngine:
    """Thermal Safety Engine."""

    # Hard constraints
    TEMP_LIMIT_MAX = 45.0          # Rack limit 
    MAX_ALLOWED_TREND = 0.5        # Max rise °C/min
    MIN_REQUIRED_HISTORY = 5       # Min data points

    def __init__(self):
        # Stateless
        pass

    def evaluate_safety(self, state: SystemStateRecord) -> SafetyStatusRecord:
        """Evaluate safety."""
        
        if not state.current_data.is_valid:
            self._veto(safety_record, "DATA_INVALID", "Cannot ensure safety on invalid telemetry.")
            return safety_record
        safety_record = SafetyStatusRecord(status=SafetyStatus.SAFE)

        # Check validation
        if not state.current_data.is_valid:
            self._veto(safety_record, "INVALID_DATA_STREAM", 
                      f"Data validation failed: {state.current_data.validation_flags}")

        # Check thermal limits
        max_current_temp = max(state.current_data.data.rack_temps)
        if max_current_temp > self.TEMP_LIMIT_MAX:
            self._veto(safety_record, "THERMAL_LIMIT_EXCEEDED", 
                      f"Rack temp {max_current_temp}°C exceeds limit {self.TEMP_LIMIT_MAX}°C")

        # Check thermal trends
        if state.window_15m_count >= self.MIN_REQUIRED_HISTORY:
            if state.delta_t_trend > self.MAX_ALLOWED_TREND:
                self._veto(safety_record, "RAPID_TEMP_RISE", 
                          f"Delta T trend {state.delta_t_trend}°C/min exceeds safe margin")
        else:
            # Block if history insufficient
            self._veto(safety_record, "INSUFFICIENT_HISTORY", 
                      "Waiting for historical buffer to stabilize safety trends")

        # Final status assignment
        if not safety_record.reason_codes:
            safety_record.description = "All thermal and data integrity checks passed."
        
        return safety_record

    def _veto(self, record: SafetyStatusRecord, code: str, desc: str):
        """Apply veto."""
        record.status = SafetyStatus.BLOCKED
        record.reason_codes.append(code)
        # Log reason
        record.description += f"[{code}: {desc}] "

# Example usage:
# safety_report = safety_engine.evaluate_safety(current_state)
# if safety_report.status == SafetyStatus.BLOCKED:
#     return "Optimization Blocked: " + safety_report.description